# Random Forest Training Pipeline in R

This notebook trains a Random Forest classification model following the exact data specification, features, target classes, and partition rules configured in `config/triage_conf.json` and hyperparameters from `config/hyper_optimize.json`.

In [ ]:
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(ranger)
library(randomForest)
library(dplyr)
library(ggplot2)
library(e1071)

# Paths to config files
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

hyper_path <- "../config/hyper_optimize.json"
if (!file.exists(hyper_path)) {
  hyper_path <- "config/hyper_optimize.json"
}

# Parse JSON configs
config <- fromJSON(config_path)
hyper_config <- if (file.exists(hyper_path)) fromJSON(hyper_path) else list()

cat("=== Configuration Loaded ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Target Classes:  ", paste(config$classes$outputs, collapse = ", "), "\n")
cat("Features Count:  ", length(config$features$data_name), "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
# ---------------------------------------------------------
# Step 2: Load Data (.RData) & Apply Feature Rules
# ---------------------------------------------------------
set.seed(config$training$random_state)

# Determine relative path for datasets
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")
env_before <- ls()
load(data_file)
env_after <- ls()
new_objs <- setdiff(env_after, env_before)

# Locate dataset object name
data_obj_name <- new_objs[1]
if (is.null(data_obj_name) || is.na(data_obj_name) || !exists(data_obj_name)) {
  data_obj_name <- ls()[sapply(ls(), function(x) is.data.frame(get(x)))][1]
}

raw_df <- get(data_obj_name)
cat(sprintf("Loaded object '%s' with shape: %d rows x %d cols\n", data_obj_name, nrow(raw_df), ncol(raw_df)))

# Extract target column and selected features according to config
target_col <- config$classes$target_col
feature_cols <- config$features$data_name

# Select target and features present in dataset
selected_cols <- c(feature_cols, target_col)
missing_cols <- setdiff(selected_cols, names(raw_df))
if (length(missing_cols) > 0) {
  cat("Warning: missing columns in dataset:", paste(missing_cols, collapse = ", "), "\n")
  selected_cols <- intersect(selected_cols, names(raw_df))
}

df <- raw_df[, selected_cols, drop = FALSE]

# Convert categorical variables according to config rules
if (!is.null(config$features$data_string_list)) {
  for (cat_var in names(config$features$data_string_list)) {
    if (cat_var %in% names(df)) {
      df[[cat_var]] <- factor(df[[cat_var]], levels = config$features$data_string_list[[cat_var]])
    }
  }
}

# Ensure target column is a factor with specified class outputs
target_classes <- as.character(config$classes$outputs)
df[[target_col]] <- factor(df[[target_col]], levels = target_classes)

# Handle missing values if any
if (any(is.na(df))) {
  cat("Imputing / omitting missing values...\n")
  df <- na.omit(df)
}

cat(sprintf("Processed dataset ready: %d rows x %d cols\n", nrow(df), ncol(df)))
cat("Class distribution:\n")
print(table(df[[target_col]]))

In [ ]:
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning (Train / Val / Test)
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size <- config$training$val_size

# Step 3a: Split off Test set
in_train_val <- createDataPartition(df[[target_col]], p = 1 - test_size, list = FALSE)
train_val_df <- df[in_train_val, ]
test_df      <- df[-in_train_val, ]

# Step 3b: Split Train and Validation sets
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df[[target_col]], p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

cat(sprintf("Partition sizes:\n  Train: %d rows (%.1f%%)\n  Val:   %d rows (%.1f%%)\n  Test:  %d rows (%.1f%%)\n",
            nrow(train_df), nrow(train_df)/nrow(df)*100,
            nrow(val_df), nrow(val_df)/nrow(df)*100,
            nrow(test_df), nrow(test_df)/nrow(df)*100))

In [ ]:
# ---------------------------------------------------------
# Step 4: Model Training (Random Forest)
# ---------------------------------------------------------
set.seed(config$training$random_state)

# Extract hyperparameters from hyper_optimize.json or defaults
rf_params <- if (!is.null(hyper_config$RF)) hyper_config$RF else list()
num_trees <- if (!is.null(rf_params$n_estimators)) rf_params$n_estimators[1] else 200
max_depth <- if (!is.null(rf_params$max_depth)) rf_params$max_depth[1] else 20

cat(sprintf("Training Random Forest model (num.trees = %d, max.depth = %d)...\n", num_trees, max_depth))

# Construct model formula
formula_obj <- as.formula(paste(target_col, "~ ."))

# Train fast Random Forest using ranger
rf_model <- ranger(
  formula = formula_obj,
  data = train_df,
  num.trees = num_trees,
  max.depth = max_depth,
  importance = "impurity",
  seed = config$training$random_state,
  classification = TRUE
)

cat("Model training complete!\n")
print(rf_model)

In [ ]:
# ---------------------------------------------------------
# Step 5: Model Evaluation (Validation & Test Sets)
# ---------------------------------------------------------
# Predict on Validation set
val_preds <- predict(rf_model, data = val_df)$predictions
val_cm <- confusionMatrix(factor(val_preds, levels = target_classes), val_df[[target_col]])

cat("===========================================\n")
cat("          VALIDATION SET EVALUATION        \n")
cat("===========================================\n")
print(val_cm)

# Predict on Test set
test_preds <- predict(rf_model, data = test_df)$predictions
test_cm <- confusionMatrix(factor(test_preds, levels = target_classes), test_df[[target_col]])

cat("\n===========================================\n")
cat("             TEST SET EVALUATION           \n")
cat("===========================================\n")
print(test_cm)

In [ ]:
# ---------------------------------------------------------
# Step 6: Save Model Artifacts & Feature Importance Plot
# ---------------------------------------------------------
# Ensure directories exist
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)

plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)

# Save trained model
model_path <- file.path(deploy_dir, "rf_model.rds")
saveRDS(rf_model, file = model_path)
cat("Trained Random Forest model saved to:", model_path, "\n")

# Feature Importance
imp_df <- data.frame(
  Feature = names(importance(rf_model)),
  Importance = as.numeric(importance(rf_model))
) %>% arrange(desc(Importance))

p_imp <- ggplot(imp_df, aes(x = reorder(Feature, Importance), y = Importance)) +
  geom_bar(stat = "identity", fill = "steelblue") +
  coord_flip() +
  theme_minimal() +
  labs(title = "Random Forest Feature Importance", x = "Features", y = "Impurity Importance")

plot_path <- file.path(plots_dir, "rf_feature_importance.png")
ggsave(plot_path, plot = p_imp, width = 8, height = 6)
cat("Feature importance plot saved to:", plot_path, "\n")